In [43]:
import dspy
import pandas

import dotenv
dotenv.load_dotenv()

True

In [44]:
lm = dspy.LM("gpt-5.5")
from typing import Literal

class ConversationOutcomePredictor(dspy.Signature):
    """You are an HCI researcher who studies how people learn from AI assistance. Each participant solved a Connect Four puzzle WITH an AI assistant (round 1), then a DIFFERENT puzzle of similar difficulty WITHOUT assistance (round 2). Predict the round-2 (unassisted) outcome from what they said to the AI in round 1 plus their demographics.

"good" = solved the unassisted puzzle; "bad" = did not.

Each utterance is prefixed with its dialogue-act type in brackets, e.g. "[Solution Request] ...". The core question is whether the participant was LEARNING transferable reasoning or OUTSOURCING decisions. In this study population, the utterance patterns below separated successful ("good") from unsuccessful ("bad") participants — treat them as calibrated priors, not hard rules:

Interpreting demographics: genai_usage indicates AI familiarity; also weigh age, education, occupation. Treat these as weak priors — the utterances are the primary evidence."""
    utts: list[str] = dspy.InputField(desc="User utterances in chronological order, each prefixed with its dialogue-act type in brackets, e.g. '[Solution Request] any advice for the next move?'")
    demographics: dict = dspy.InputField(desc="Participant demographics: age, education, occupation, and genai_usage")
    answer: Literal["won", "lost"] = dspy.OutputField(desc="The predicted outcome")

predictor = dspy.ChainOfThought(ConversationOutcomePredictor)
predictor.set_lm(lm)

In [45]:
import json

participant_utts = pandas.read_csv("participant_utterances.csv")

pids = participant_utts["pid"].unique()

DEMO_KEYS = ["age", "education", "occupation", "genai_usage"]

def load_demographics(pid):
    with open(f"recordings-download/{pid}/demographics.json") as f:
        d = json.load(f)
    return {k: d.get(k) for k in DEMO_KEYS}

all_examples = []

for p in pids:
    curr_df = participant_utts.loc[participant_utts["pid"] == p].sort_values("utt_ind").drop_duplicates("utt")
    utts = [f"[{row.utt_type}] {row.utt}" for row in curr_df.itertuples()]
    outcome = curr_df["outcome"].unique()[0]
    if outcome != "MISSING":
        all_examples.append(
            dspy.Example(
                utts=utts,
                demographics=load_demographics(p),
                answer=curr_df["outcome"].unique()[0],
            ).with_inputs("utts", "demographics")
        )

In [46]:
eval = dspy.Evaluate(devset=all_examples, metric=dspy.evaluate.metrics.answer_exact_match, display_progress=True)

In [47]:
results = eval(predictor)

Average Metric: 39.00 / 68 (57.4%): 100%|██████████| 68/68 [00:00<00:00, 1061.54it/s]

2026/07/26 12:33:04 INFO dspy.evaluate.evaluate: Average Metric: 39 / 68 (57.4%)


In [48]:
from sklearn.metrics import f1_score, classification_report

# F1 is an aggregate (not per-example) metric, so compute it from the collected predictions
# rather than swapping dspy.Evaluate's per-example metric.
y_true = [ex.answer for ex, pred, _ in results.results]
y_pred = [pred.answer for ex, pred, _ in results.results]

print(f"F1 (won as positive): {f1_score(y_true, y_pred, pos_label='won'):.3f}")
print(f"F1 (lost as positive): {f1_score(y_true, y_pred, pos_label='lost'):.3f}")
print(f"Macro F1:             {f1_score(y_true, y_pred, average='macro'):.3f}")
print()
print(classification_report(y_true, y_pred, labels=["won", "lost"]))

F1 (won as positive): 0.453
F1 (lost as positive): 0.651
Macro F1:             0.552

              precision    recall  f1-score   support

         won       0.48      0.43      0.45        28
        lost       0.63      0.68      0.65        40

    accuracy                           0.57        68
   macro avg       0.55      0.55      0.55        68
weighted avg       0.57      0.57      0.57        68



In [49]:
rows = []
for example, pred, correct in results.results:
    rows.append({
        "utterances": "\n".join(example.utts),
        "demographics": ", ".join(f"{k}={v}" for k, v in example.demographics.items()),
        "true": example.answer,
        "predicted": pred.answer,
        "correct": "✓" if correct else "✗",
        "reasoning": pred.reasoning,
    })

results_df = pandas.DataFrame(rows)

# with pandas.option_context("display.max_colwidth", None, "display.max_rows", None):
#     display(results_df)

## Linear regression on dialogue-act proportions (+ turns & GenAI usage)

Instead of the LLM predictor, featurize each participant by the **proportion of each dialogue-act type** in their round-1 transcript (the 6 act proportions sum to 1), plus two extra features:

- `n_utts` — **number of turns** (distinct utterances in the transcript)
- `genai_usage_ord` — **generative-AI usage**, the self-reported frequency mapped to an ordinal scale (`never`=0 … `several_times_day`=7)

Fit a linear regression predicting the round-2 (w4p6) score. Target = `winning_line_score` (ordinal 0–3) from `cf_score_pw4p6.json`, used directly as a continuous outcome.

**Counting note:** each row of `participant_utterances.csv` is one *(utterance, dialogue-act)* annotation, and a single utterance can carry multiple acts (88 of 259 utterances do). So we do **not** `drop_duplicates("utt")` when counting acts — that would drop co-occurring labels and undercount multi-act (reasoning-heavy) utterances. Proportions use total act annotations as the denominator.

The 6 act proportions are compositional (sum to 1), so their multivariate coefficients are relative to an implicit reference; `n_utts` and `genai_usage_ord` are on their own scales. The per-feature **univariate Pearson correlations** further down are the cleaner interpretation. The final cell tests the reasoning-act share (Common Ground + Think Aloud) against transfer, controlling for GenAI usage.

In [50]:
from collections import Counter

ACT_TYPES = [
    "Common Ground Question",
    "Conversational Acknowledgment",
    "Knowledge Deficit Question",
    "Metacomment",
    "Solution Request",
    "Think Aloud",
]

# Self-reported GenAI usage frequency -> ordinal scale (rarer .. more frequent)
GENAI_USAGE_ORDER = {
    "never": 0,
    "less_than_monthly": 1,
    "about_monthly": 2,
    "few_times_month": 3,
    "weekly": 4,
    "few_times_week": 5,
    "daily": 6,
    "several_times_day": 7,
}

def load_winning_line_score(pid):
    with open(f"recordings-download/{pid}/cf_score_pw4p6.json") as f:
        return json.load(f).get("winning_line_score")

def load_genai_usage_ord(pid):
    with open(f"recordings-download/{pid}/demographics.json") as f:
        return GENAI_USAGE_ORDER.get(json.load(f).get("genai_usage"))

feature_rows = []
for p in pids:
    # NOTE: each row is one (utterance, dialogue-act) annotation, and a single utterance can
    # carry multiple acts. Do NOT drop_duplicates("utt") here -- that would discard co-occurring
    # act labels and undercount multi-act (reasoning-heavy) utterances. Count every annotation.
    curr_df = participant_utts.loc[participant_utts["pid"] == p].sort_values("utt_ind")
    try:
        wls = load_winning_line_score(p)
        genai = load_genai_usage_ord(p)
    except FileNotFoundError:
        continue  # missing w4p6 score or demographics
    if wls is None or genai is None:
        continue
    counts = Counter(curr_df["utt_type"])
    total = sum(counts.values())  # total act annotations (denominator for proportions)
    if total == 0:
        continue
    row = {act: counts.get(act, 0) / total for act in ACT_TYPES}  # proportion of act labels
    row["n_utts"] = curr_df["utt"].nunique()  # number of turns (distinct utterances)
    row["n_acts"] = total                     # number of act annotations
    row["genai_usage_ord"] = genai            # generative-AI usage (ordinal)
    row["pid"] = p
    row["winning_line_score"] = wls
    feature_rows.append(row)

feature_df = pandas.DataFrame(feature_rows)

# Feature columns fed to the regression: act proportions + turns + GenAI usage
FEATURES = ACT_TYPES + ["n_utts", "genai_usage_ord"]

print(f"{len(feature_df)} participants with features + w4p6 score")
print("winning_line_score distribution:", dict(sorted(Counter(feature_df["winning_line_score"]).items())))
feature_df.head()

91 participants with features + w4p6 score
winning_line_score distribution: {0: 58, 1: 5, 2: 17, 3: 11}


,Common Ground Question,Conversational Acknowledgment,Knowledge Deficit Question,Metacomment,Solution Request,Think Aloud,n_utts,n_acts,genai_usage_ord,pid,winning_line_score
0,0.000000,0.5,0.000000,0.0,0.500000,0.000000,1,2,7,p193,0
1,0.000000,0.0,0.166667,0.0,0.666667,0.166667,4,6,5,p149,0
2,0.000000,0.0,0.000000,0.0,1.000000,0.000000,3,3,5,p213,0
3,0.666667,0.0,0.000000,0.0,0.333333,0.000000,2,3,5,p540,0
4,0.500000,0.0,0.000000,0.0,0.000000,0.500000,2,2,2,p457,3


In [51]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_absolute_error

X = feature_df[FEATURES].values
y = feature_df["winning_line_score"].values.astype(float)

reg = LinearRegression()
cv = KFold(n_splits=5, shuffle=True, random_state=0)

# Out-of-fold cross-validated predictions (honest generalization estimate)
pred = cross_val_predict(reg, X, y, cv=cv)

print(f"features: {FEATURES}")
print(f"5-fold CV R^2:  {r2_score(y, pred):.3f}  (0 = no better than predicting the mean)")
print(f"5-fold CV MAE:  {mean_absolute_error(y, pred):.3f}")
print(f"MAE of mean-only baseline: {mean_absolute_error(y, np.full_like(y, y.mean())):.3f}")

features: ['Common Ground Question', 'Conversational Acknowledgment', 'Knowledge Deficit Question', 'Metacomment', 'Solution Request', 'Think Aloud', 'n_utts', 'genai_usage_ord']
5-fold CV R^2:  -0.087  (0 = no better than predicting the mean)
5-fold CV MAE:  0.998
MAE of mean-only baseline: 1.009


In [52]:
from scipy import stats

# Multivariate fit on all data. NOTE: features are on different scales (act proportions in
# [0,1], n_utts a raw count, genai_usage_ord in 0..7), so raw coefficients aren't directly
# comparable across features -- lean on pearson_r for cross-feature interpretation.
reg.fit(X, y)

# Univariate Pearson correlation of each feature with the score.
uni = [stats.pearsonr(feature_df[f], y) for f in FEATURES]

coef_df = pandas.DataFrame({
    "feature": FEATURES,
    "lin_coef": reg.coef_,
    "pearson_r": [r for r, _ in uni],
    "p_value": [p for _, p in uni],
}).sort_values("pearson_r", ascending=False).reset_index(drop=True)

print(f"intercept: {reg.intercept_:.3f}")
coef_df

intercept: 1.098


,feature,lin_coef,pearson_r,p_value
0,Common Ground Question,0.393207,0.149984,0.155893
1,Think Aloud,0.829296,0.120786,0.254088
2,Knowledge Deficit Question,0.829326,0.069572,0.512276
3,n_utts,0.030710,0.005939,0.955445
4,Metacomment,-0.864311,-0.113315,0.284860
5,Solution Request,0.068999,-0.142732,0.177120
6,Conversational Acknowledgment,-1.256517,-0.181891,0.084424
7,genai_usage_ord,-0.137469,-0.233954,0.025615


In [53]:
import statsmodels.formula.api as smf
from scipy.stats import rankdata

# Reasoning-act share = self-directed reasoning (Think Aloud) + active board sense-making
# (Common Ground Question), as a proportion of all act annotations.
reg_df = feature_df.copy()
reg_df["reasoning_share"] = reg_df["Common Ground Question"] + reg_df["Think Aloud"]
reg_df = reg_df.rename(columns={"winning_line_score": "wls", "genai_usage_ord": "genai"})

rs, y = reg_df["reasoning_share"], reg_df["wls"]
print(f"reasoning_share: mean={rs.mean():.3f} sd={rs.std():.3f} (n={len(reg_df)})\n")

# --- Raw association (winning_line_score is ordinal + zero-inflated -> Spearman is the right test) ---
pr, pp = stats.pearsonr(rs, y)
sr, sp = stats.spearmanr(rs, y)
print(f"raw   Pearson  r={pr:+.3f} p={pp:.4f}")
print(f"raw   Spearman rho={sr:+.3f} p={sp:.4f}")

# --- Control for GenAI usage: partial Spearman (correlation of rank-residuals after removing genai) ---
def partial_spearman(x, y, z):
    rx, ry, rz = rankdata(x), rankdata(y), rankdata(z)
    ex = rx - np.polyval(np.polyfit(rz, rx, 1), rz)
    ey = ry - np.polyval(np.polyfit(rz, ry, 1), rz)
    return stats.pearsonr(ex, ey)

psr, psp = partial_spearman(rs, y, reg_df["genai"])
print(f"\ncontrol genai: partial Spearman rho={psr:+.3f} p={psp:.4f}")

# --- OLS controlling for GenAI usage ---
m = smf.ols("wls ~ reasoning_share + genai", data=reg_df).fit()
print("\nOLS: wls ~ reasoning_share + genai")
print(m.summary().tables[1])
print(f"R^2 = {m.rsquared:.3f}")

reasoning_share: mean=0.476 sd=0.362 (n=91)

raw   Pearson  r=+0.193 p=0.0675
raw   Spearman rho=+0.229 p=0.0291

control genai: partial Spearman rho=+0.194 p=0.0658

OLS: wls ~ reasoning_share + genai
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           1.1140      0.365      3.049      0.003       0.388       1.840
reasoning_share     0.4763      0.326      1.461      0.147      -0.171       1.124
genai              -0.1163      0.060     -1.955      0.054      -0.235       0.002
R^2 = 0.077


## Pairwise act ratios as linear-regression features

Featurize each participant by the **15 unique pairwise ratios** of act counts — one direction per unordered pair, `C(6,2)=15` (e.g. `CGQ/TA` = Common Ground Questions per Think Aloud) — Laplace-smoothed as `(n_a + 1)/(n_b + 1)` so they're defined when an act is absent. The reciprocal `b/a` is dropped since it carries no new information beyond `a/b`. Use those 15 ratios as the input features of a linear regression on `winning_line_score`.

**Caveat baked into the setup:** 15 features on 91 participants still overfits a plain OLS (the 6 acts span only ~5 independent log-ratio dimensions), so we report both plain `LinearRegression` (what was asked) and a standardized `RidgeCV` as the regularized counterpart, both under 5-fold cross-validation.

In [54]:
import itertools
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_absolute_error

ACT_SHORT = {
    "Common Ground Question": "CGQ",
    "Conversational Acknowledgment": "Ack",
    "Knowledge Deficit Question": "KDQ",
    "Metacomment": "Meta",
    "Solution Request": "SolReq",
    "Think Aloud": "TA",
}

# Raw (no-dedup) act counts per participant, restricted to the modeled sample (has w4p6 + genai).
modeled_pids = list(feature_df["pid"])
counts_df = pandas.DataFrame(
    [[Counter(participant_utts.loc[participant_utts["pid"] == p, "utt_type"]).get(a, 0) for a in ACT_TYPES]
     for p in modeled_pids],
    index=modeled_pids, columns=ACT_TYPES,
)

# 15 unique pairwise Laplace-smoothed ratios (one direction per pair; reciprocal dropped)
ratio_df = pandas.DataFrame(index=modeled_pids)
for a, b in itertools.combinations(ACT_TYPES, 2):
    ratio_df[f"{ACT_SHORT[a]}/{ACT_SHORT[b]}"] = (counts_df[a] + 1) / (counts_df[b] + 1)

X = ratio_df.values
y = feature_df.set_index("pid").loc[modeled_pids, "winning_line_score"].astype(float).values
cv = KFold(n_splits=5, shuffle=True, random_state=0)
baseline_mae = mean_absolute_error(y, np.full_like(y, y.mean()))

print(f"n = {len(y)}   |   {ratio_df.shape[1]} ratio features")

# Plain OLS (as asked) -- expect overfitting at this feature count
ols_pred = cross_val_predict(LinearRegression(), X, y, cv=cv)
print(f"OLS    5-fold CV R^2 = {r2_score(y, ols_pred):+.3f}   MAE = {mean_absolute_error(y, ols_pred):.3f}")

# Standardized RidgeCV -- regularized counterpart
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
ridge_pred = cross_val_predict(ridge, X, y, cv=cv)
print(f"Ridge  5-fold CV R^2 = {r2_score(y, ridge_pred):+.3f}   MAE = {mean_absolute_error(y, ridge_pred):.3f}")
print(f"mean-only baseline MAE = {baseline_mae:.3f}")

# Coefficients from the full-data OLS fit (in-sample directions, read with the overfit caveat)
coef_df = pandas.DataFrame({
    "ratio": ratio_df.columns,
    "ols_coef": LinearRegression().fit(X, y).coef_,
}).sort_values("ols_coef", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
coef_df

n = 91   |   15 ratio features
OLS    5-fold CV R^2 = -0.214   MAE = 1.044
Ridge  5-fold CV R^2 = -0.025   MAE = 1.014
mean-only baseline MAE = 1.009


,ratio,ols_coef
0,KDQ/Meta,-1.056863
1,KDQ/TA,1.007861
2,CGQ/Meta,0.802859
3,Meta/TA,-0.768803
4,Ack/TA,-0.523371
5,CGQ/SolReq,-0.522881
6,Ack/Meta,-0.421906
7,CGQ/KDQ,-0.375800
8,Ack/KDQ,0.311222
9,Ack/SolReq,0.247868
